In [1]:
import pandas as pd
import numpy as np

print("Movie Recommendation System")
print("Libraries loaded successfully!")


Movie Recommendation System
Libraries loaded successfully!


In [3]:
import pandas as pd

df = pd.read_csv("../data/movies.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (1284, 10)

Columns:
['imdbId', 'title', 'releaseYear', 'releaseDate', 'genre', 'writers', 'actors', 'directors', 'sequel', 'hitFlop']


,imdbId,title,releaseYear,releaseDate,genre,writers,actors,directors,sequel,hitFlop
0,tt0118578,Albela,2001,20-Apr-01,Romance,Honey Irani (screenplay) | Honey Irani (story)...,Govinda | Aishwarya Rai Bachchan | Jackie Shro...,Deepak Sareen,0.0,2
1,tt0169102,Lagaan: Once Upon a Time in India,2001,8-May-02,Adventure | Drama | Musical,Ashutosh Gowariker (story) | Ashutosh Gowarike...,Aamir Khan | Gracy Singh | Rachel Shelley | Pa...,Ashutosh Gowariker,0.0,6
2,tt0187279,Meri Biwi Ka Jawab Nahin,2004,2-Jul-04,Action | Comedy,NaN,Akshay Kumar | Sridevi | Gulshan Grover | Laxm...,Pankaj Parashar | S.M. Iqbal,0.0,1
3,tt0222024,Hum Tumhare Hain Sanam,2002,24-May-02,Drama | Romance,K.S. Adiyaman | Arun Kumar (assistant dialogue...,Shah Rukh Khan | Madhuri Dixit | Salman Khan |...,K.S. Adiyaman,0.0,4
4,tt0227194,One 2 Ka 4,2001,30-Mar-01,Action | Comedy | Drama,Sanjay Chhel | Raaj Kumar Dahima (screenplay) ...,Shah Rukh Khan | Juhi Chawla | Jackie Shroff |...,Shashilal K. Nair,0.0,1


In [6]:
df = df.drop_duplicates()
df = df.dropna(subset=["title", "description", "genre"])
df = df.reset_index(drop=True)

print("Clean movies:", len(df))

KeyError: ['description']

In [7]:
print(df.columns.tolist())

['imdbId', 'title', 'releaseYear', 'releaseDate', 'genre', 'writers', 'actors', 'directors', 'sequel', 'hitFlop']


In [8]:
df = df.drop_duplicates()
df = df.dropna(subset=["title", "genre", "actors", "directors"])
df = df.reset_index(drop=True)

print("Movies:", len(df))

Movies: 1275


In [9]:
df["features"] = (
    df["genre"].fillna("") + " " +
    df["actors"].fillna("") + " " +
    df["directors"].fillna("") + " " +
    df["writers"].fillna("")
)

df[["title", "features"]].head()

,title,features
0,Albela,Romance Govinda | Aishwarya Rai Bachchan | Jac...
1,Lagaan: Once Upon a Time in India,Adventure | Drama | Musical Aamir Khan | Gracy...
2,Meri Biwi Ka Jawab Nahin,Action | Comedy Akshay Kumar | Sridevi | Gulsh...
3,Hum Tumhare Hain Sanam,Drama | Romance Shah Rukh Khan | Madhuri Dixit...
4,One 2 Ka 4,Action | Comedy | Drama Shah Rukh Khan | Juhi ...


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["features"])

similarity = cosine_similarity(tfidf_matrix)

print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (1275, 3496)


In [11]:
def recommend_movies(movie_title, n=5):
    matches = df[df["title"].str.lower() == movie_title.lower()]

    if matches.empty:
        return "Movie not found"

    idx = matches.index[0]

    scores = list(enumerate(similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]

    return df.iloc[[i for i, score in scores]][
        ["title", "releaseYear", "genre", "actors", "directors"]
    ]

In [12]:
recommend_movies(df["title"].iloc[0])

,title,releaseYear,genre,actors,directors
113,Armaan,2003,Drama | Family | Romance,Amitabh Bachchan | Anil Kapoor | Preity Zinta ...,Honey Irani
71,Kya Yehi Pyaar Hai,2002,Comedy | Musical | Romance,Jackie Shroff | Aftab Shivdasani | Ameesha Pat...,K. Muralimohana Rao
305,Krrish,2006,Action | Adventure | Romance,Rekha | Hrithik Roshan | Priyanka Chopra | Sha...,Rakesh Roshan
55,Tera Mera Saath Rahen,2001,Drama,Ajay Devgn | Sonali Bendre | Namrata Shirodkar...,Mahesh Manjrekar
448,Sarkar Raj,2008,Action | Crime | Drama,Amitabh Bachchan | Abhishek Bachchan | Aishwar...,Ram Gopal Varma


In [13]:
df.to_csv("../data/clean_movies.csv", index=False)

print("Saved:", len(df), "movies")

Saved: 1275 movies


In [14]:
print(df.columns.tolist())


['imdbId', 'title', 'releaseYear', 'releaseDate', 'genre', 'writers', 'actors', 'directors', 'sequel', 'hitFlop', 'features']


In [15]:
print(df.head(2).to_dict("records"))

[{'imdbId': 'tt0118578', 'title': 'Albela', 'releaseYear': 2001, 'releaseDate': '20-Apr-01', 'genre': 'Romance', 'writers': 'Honey Irani (screenplay) | Honey Irani (story) | Javed Siddiqui (dialogue)', 'actors': 'Govinda | Aishwarya Rai Bachchan | Jackie Shroff | Namrata Shirodkar', 'directors': 'Deepak Sareen', 'sequel': 0.0, 'hitFlop': 2, 'features': 'Romance Govinda | Aishwarya Rai Bachchan | Jackie Shroff | Namrata Shirodkar Deepak Sareen Honey Irani (screenplay) | Honey Irani (story) | Javed Siddiqui (dialogue)'}, {'imdbId': 'tt0169102', 'title': 'Lagaan: Once Upon a Time in India', 'releaseYear': 2001, 'releaseDate': '8-May-02', 'genre': 'Adventure | Drama | Musical', 'writers': 'Ashutosh Gowariker (story) | Ashutosh Gowariker (screenplay) | Kumar Dave (screenplay) | Sanjay Dayma (screenplay) | K.P. Saxena (dialogue)', 'actors': 'Aamir Khan | Gracy Singh | Rachel Shelley | Paul Blackthorne', 'directors': 'Ashutosh Gowariker', 'sequel': 0.0, 'hitFlop': 6, 'features': 'Adventure | 